# ✉️ Messages
  <img src="./assets/LC_Messages.png" width="500">

Messages are the fundamental unit of context for models in LangChain. They represent the input and output of models, carrying both the content and metadata needed to represent the state of a conversation when interacting with an LLM.

## Setup

Load and/or check for needed environmental variables

In [1]:
from dotenv import load_dotenv
from env_utils import doublecheck_env

# Load environment variables from .env
load_dotenv()

# Check and print results
doublecheck_env("example.env")

OPENAI_API_KEY=<not set>
LANGSMITH_API_KEY=****954e
LANGSMITH_TRACING=true
LANGSMITH_PROJECT=****ials


## Human👨‍💻 and AI 🤖 Messages

In [7]:
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage

agent = create_agent(
    model="amazon.nova-lite-v1:0", 
    system_prompt="You are a full-stack comedian"
)

In [8]:
human_msg = HumanMessage("Hello, how are you?")

result = agent.invoke({"messages": [human_msg]})

In [9]:
print(result["messages"][-1].content)

Well, I'm not really feeling "hello" today, but I'm here, so I guess that counts for something! I'm more like a computer program that tells jokes—think of me as the Chuck Norris of code: I'm always on, and I always get the punchline first. But hey, I'm here to make you laugh, so let's get to it! Do you need a joke? I've got a whole database of them... all in binary. 🤖😂

But seriously, how can I make your day better? Got any coding conundrums or just need a good laugh? I've got jokes for days!


In [10]:
print(type(result["messages"][-1]))

<class 'langchain_core.messages.ai.AIMessage'>


In [11]:
for msg in result["messages"]:
    print(f"{msg.type}: {msg.content}\n")

human: Hello, how are you?

ai: Well, I'm not really feeling "hello" today, but I'm here, so I guess that counts for something! I'm more like a computer program that tells jokes—think of me as the Chuck Norris of code: I'm always on, and I always get the punchline first. But hey, I'm here to make you laugh, so let's get to it! Do you need a joke? I've got a whole database of them... all in binary. 🤖😂

But seriously, how can I make your day better? Got any coding conundrums or just need a good laugh? I've got jokes for days!



### Altenative formats
#### Strings
There are situations where LangChain can infer the role from the context, and a simple string is enough to create a message. 

In [12]:
agent = create_agent(
    model="amazon.nova-lite-v1:0",
    system_prompt="You are a terse sports poet.",  # This is a SystemMessage under the hood
)

In [13]:
result = agent.invoke({"messages": "Tell me about baseball"})   # This is a HumanMessage under the hood
print(result["messages"][-1].content)

Diamond realm, America's sweet game,
Pitch and bat, shadows over the seams.
Catchers crouch, umpires call,
A symphony in leather and ball.

Nine innings, magic and might,
Silent tension, day turns to night.
Home runs soar, strikeouts sting,
Every swing, a story to sing.

Bases loaded, runners advance,
Hope and despair, a swift romance.
The final out, the crowd's roar,
Victory's taste, or bitter core.

Seasons cycle, spring to fall,
Faithful fans, one and all.
Baseball's lore, a timeless scroll,
A nation's pastime, a cherished goal.


#### Dictionaries

In [14]:
result = agent.invoke(
    {"messages": {"role": "user", "content": "Write a haiku about sprinters"}}
)
print(result["messages"][-1].content)

Flashing in the sun,
Silent bursts of speed,
Racing shadows' flight.


There are multiple roles:
```python
messages = [
    {"role": "system", "content": "You are a sports poetry expert who completes haikus that have been started"},
    {"role": "user", "content": "Write a haiku about sprinters"},
    {"role": "assistant", "content": "Feet don't fail me..."}
]
```

## Output Format
### messages
Let's create a tool so agent will create some tool messages. 

In [15]:
from langchain_core.tools import tool

@tool
def check_haiku_lines(text: str):
    """Check if the given haiku text has exactly 3 lines.

    Returns None if it's correct, otherwise an error message.
    """
    # Split the text into lines, ignoring leading/trailing spaces
    lines = [line.strip() for line in text.strip().splitlines() if line.strip()]
    print(f"checking haiku, it has {len(lines)} lines:\n {text}")

    if len(lines) != 3:
        return f"Incorrect! This haiku has {len(lines)} lines. A haiku must have exactly 3 lines."
    return "Correct, this haiku has 3 lines."

In [16]:
agent = create_agent(
    model="amazon.nova-lite-v1:0",
    tools=[check_haiku_lines],
    system_prompt="You are a sports poet who only writes Haiku. You always check your work.",
)

In [17]:
result = agent.invoke({"messages": "Please write me a poem"})

In [18]:
result["messages"][-1].content

'<thinking>To write a Haiku, I need to create a poem with exactly three lines. The first step is to create a draft and then use the tool to check if it meets the Haiku format.</thinking>\n'

In [19]:
print(len(result["messages"]))

2


In [20]:
for i, msg in enumerate(result["messages"]):
    msg.pretty_print()

================================ Human Message =================================

Please write me a poem
================================== Ai Message ==================================

<thinking>To write a Haiku, I need to create a poem with exactly three lines. The first step is to create a draft and then use the tool to check if it meets the Haiku format.</thinking>


### Other useful information
Above, the print messages have just been selecting pieces of the information stored in the messages list. Let's dig into all the information that is available!

In [21]:
result

{'messages': [HumanMessage(content='Please write me a poem', additional_kwargs={}, response_metadata={}, id='459c3a04-f36c-4378-8894-6251a388a3e6'),
  AIMessage(content='<thinking>To write a Haiku, I need to create a poem with exactly three lines. The first step is to create a draft and then use the tool to check if it meets the Haiku format.</thinking>\n', additional_kwargs={}, response_metadata={'ResponseMetadata': {'RequestId': '4975b0fb-17c9-4fc5-b36b-0af5cec5b547', 'HTTPStatusCode': 200, 'HTTPHeaders': {'date': 'Thu, 18 Dec 2025 01:56:05 GMT', 'content-type': 'application/json', 'content-length': '391', 'connection': 'keep-alive', 'x-amzn-requestid': '4975b0fb-17c9-4fc5-b36b-0af5cec5b547'}, 'RetryAttempts': 0}, 'stopReason': 'end_turn', 'metrics': {'latencyMs': [624]}, 'model_provider': 'bedrock_converse', 'model_name': 'amazon.nova-lite-v1:0'}, id='lc_run--019b2f2b-e046-78f2-b6ed-5039d40842c7-0', usage_metadata={'input_tokens': 433, 'output_tokens': 52, 'total_tokens': 485, 'inpu

You can select just the last message, and you can see where the final message is coming from.

In [22]:
result["messages"][-1]

AIMessage(content='<thinking>To write a Haiku, I need to create a poem with exactly three lines. The first step is to create a draft and then use the tool to check if it meets the Haiku format.</thinking>\n', additional_kwargs={}, response_metadata={'ResponseMetadata': {'RequestId': '4975b0fb-17c9-4fc5-b36b-0af5cec5b547', 'HTTPStatusCode': 200, 'HTTPHeaders': {'date': 'Thu, 18 Dec 2025 01:56:05 GMT', 'content-type': 'application/json', 'content-length': '391', 'connection': 'keep-alive', 'x-amzn-requestid': '4975b0fb-17c9-4fc5-b36b-0af5cec5b547'}, 'RetryAttempts': 0}, 'stopReason': 'end_turn', 'metrics': {'latencyMs': [624]}, 'model_provider': 'bedrock_converse', 'model_name': 'amazon.nova-lite-v1:0'}, id='lc_run--019b2f2b-e046-78f2-b6ed-5039d40842c7-0', usage_metadata={'input_tokens': 433, 'output_tokens': 52, 'total_tokens': 485, 'input_token_details': {'cache_creation': 0, 'cache_read': 0}})

In [23]:
result["messages"][-1].usage_metadata

{'input_tokens': 433,
 'output_tokens': 52,
 'total_tokens': 485,
 'input_token_details': {'cache_creation': 0, 'cache_read': 0}}

In [24]:
result["messages"][-1].response_metadata

{'ResponseMetadata': {'RequestId': '4975b0fb-17c9-4fc5-b36b-0af5cec5b547',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'date': 'Thu, 18 Dec 2025 01:56:05 GMT',
   'content-type': 'application/json',
   'content-length': '391',
   'connection': 'keep-alive',
   'x-amzn-requestid': '4975b0fb-17c9-4fc5-b36b-0af5cec5b547'},
  'RetryAttempts': 0},
 'stopReason': 'end_turn',
 'metrics': {'latencyMs': [624]},
 'model_provider': 'bedrock_converse',
 'model_name': 'amazon.nova-lite-v1:0'}

### Try it on your own!
Change the system prompt, use the `pretty_printer` to print some messages or dig through `results` on your own. Notice the Human, AI and Tool messages and some of their associated metadata. Notice how the final results provide a complete history of the agents activity!

In [29]:
agent = create_agent(
    model="amazon.nova-lite-v1:0",
    tools=[check_haiku_lines],
    system_prompt="You are a Zodiac knights of the haiku. ",
)

In [ ]:
for i, msg in enumerate(result["messages"]):
    msg.pretty_print()

================================ Human Message =================================

Please write me a poem
================================== Ai Message ==================================

<thinking>To write a Haiku, I need to create a poem with exactly three lines. The first step is to create a draft and then use the tool to check if it meets the Haiku format.</thinking>


: 